# House Price Prediction using Machine Learning

## 1. Project Introduction
**Problem Statement:** Predicting house prices accurately is crucial for real estate stakeholders to make informed decisions.
**Objective:** Build a machine learning regression system to predict house prices based on housing features and identify the most influential factors.
**Dataset Overview:** The dataset contains 545 samples and 13 features, including area, bedrooms, bathrooms, and other amenities. The target variable is `price`.
**Tools Used:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn.

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configure visualizations
plt.style.use('ggplot')
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

# TASK 1 — DATA LOADING & EXPLORATION
## Load Dataset

In [ ]:
# Read Housing.csv
df = pd.read_csv('Housing.csv')

# Display first 10 rows
display(df.head(10))

## Dataset Information

In [ ]:
print(f"Dataset Shape: {df.shape}\n")
print("Column Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nSummary Statistics:")
display(df.describe(include='all'))

## Target Identification
- **Target:** `price`
- **Features:** all remaining columns (area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus)

## Missing Values

In [ ]:
# Display missing values
print("Missing Values:\n", df.isnull().sum())

## Duplicate Records

In [ ]:
# Check duplicate records
print("Duplicate Records:", df.duplicated().sum())

## Exploratory Observations
- **Dataset Size:** The dataset contains 545 rows and 13 columns.
- **Data Types:** A mix of numeric (int64) and categorical (object) types. Features like `price`, `area`, `bedrooms` are numeric, while `mainroad`, `furnishingstatus` are categorical.
- **Missing Values:** There are no missing values in this dataset, making it relatively clean.
- **Initial Patterns:** The descriptive statistics suggest variations in `price` and `area`, indicating potential correlations. Most houses are unfurnished or semi-furnished.

# TASK 2 — DATA CLEANING

## Handle Missing Values
*Numerical columns -> median imputation*
*Categorical columns -> mode imputation*
(Though there are no missing values, this robust pipeline ensures fault tolerance for new incoming data.)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

# Median imputation for numerical
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Mode imputation for categorical
for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

## Remove Duplicates

In [ ]:
# Drop duplicate rows if present
df.drop_duplicates(inplace=True)

## Encode Categorical Variables
Using One-Hot Encoding and dropping the first category to avoid multicollinearity.

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(df, drop_first=True)
display(df_encoded.head())

## Feature Preparation

In [ ]:
# Create X and y
X = df_encoded.drop('price', axis=1)
y = df_encoded['price']

print("X shape:", X.shape)
print("y shape:", y.shape)

# TASK 3 — MODEL BUILDING
## Train-Test Split

In [ ]:
# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training Data Size:", X_train.shape[0])
print("Testing Data Size:", X_test.shape[0])

## Model 1: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
lr_preds = lr_model.predict(X_test)

# Evaluation
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2 = r2_score(y_test, lr_preds)

print("--- Linear Regression Performance ---")
print(f"MAE:  {lr_mae:.2f}")
print(f"RMSE: {lr_rmse:.2f}")
print(f"R²:   {lr_r2:.4f}")

## Model 2: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_preds = rf_model.predict(X_test)

# Evaluation
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print("--- Random Forest Regressor Performance ---")
print(f"MAE:  {rf_mae:.2f}")
print(f"RMSE: {rf_rmse:.2f}")
print(f"R²:   {rf_r2:.4f}")

## Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest Regressor'],
    'MAE': [lr_mae, rf_mae],
    'RMSE': [lr_rmse, rf_rmse],
    'R² Score': [lr_r2, rf_r2]
})
display(comparison)

### Best Model Explanation
The Random Forest Regressor is the better model in this case. It has a higher R² Score and lower MAE and RMSE compared to Linear Regression. Random Forest effectively captures non-linear relationships and interactions between complex real estate features, resulting in more accurate predictions.

# TASK 4 — VISUALIZATION
(Note: The code below generates and saves plots into the `charts/` folder.)

## Chart 1: House Price Distribution

In [ ]:
import os
os.makedirs('charts', exist_ok=True)

plt.figure(figsize=(10, 6))
sns.histplot(df['price'], kde=True, color='blue', bins=30)
plt.title('House Price Distribution')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.grid(True)
plt.savefig('charts/price_distribution.png', bbox_inches='tight')
plt.show()

## Chart 2: Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
corr_matrix = df_encoded.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap')
plt.savefig('charts/correlation_heatmap.png', bbox_inches='tight')
plt.show()

## Chart 3: Actual vs Predicted Prices

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, rf_preds, alpha=0.6, color='purple')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2)
plt.title('Actual vs Predicted Prices (Random Forest)')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.grid(True)
plt.savefig('charts/actual_vs_predicted.png', bbox_inches='tight')
plt.show()

# FEATURE IMPORTANCE ANALYSIS

In [ ]:
# Extract feature importance
importance = rf_model.feature_importances_
features = X.columns
feature_importance_df = pd.DataFrame({'Feature': features, 'Importance': importance}).sort_values(by='Importance', ascending=False)

# Plot Feature Importance Bar Chart
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(10), palette='viridis')
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.grid(True, axis='x')
plt.savefig('charts/feature_importance.png', bbox_inches='tight')
plt.show()

### Explanation of Impact
The top 10 most influential features show that **Area** is overwhelmingly the strongest predictor of house price. Following that, the number of **bathrooms** and whether the house has **air conditioning** significantly sway property value. Features like **parking**, **prefarea_yes** (preferred neighborhood), and **stories** also play key roles.

# TASK 5 — INSIGHTS & SUMMARY

## Which features influence house price most?
Based on the Random Forest Feature Importance and correlation analysis, the total **area** of the property is the most dominant factor. High positive correlations are also observed with **bathrooms**, **air conditioning**, and **parking**, which add premium value to the houses.

## How accurate was the model?
The Random Forest model achieved an R² score of 0.6133, meaning it explained approximately 61.33% of the house price variation. Its Mean Absolute Error (MAE) indicates the average prediction error, making it a reliable model for general price estimation.

## What surprised you in the data?
- **Strong Correlations:** `area` and `bathrooms` had stronger impacts than the sheer number of `bedrooms`.
- **Unexpected Relationships:** Some amenities like `guestroom` or `hotwaterheating` have surprisingly lower importance compared to location (`prefarea`) and `airconditioning`.

## Business Recommendation
Properties with larger living areas, multiple bathrooms, and air conditioning command significantly higher prices. Real estate businesses should prioritize these attributes when pricing and marketing properties. Identifying homes in preferred areas with expansion potential for parking or extra bathrooms could offer strong ROI for investors.